# Objective

Explore the World Bank API to understand the available economic indicators for SMP.

## Questions

- Which indicators are useful?
- Which countries are available?
- How should dates be stored?
- What cleaning is required?

## Outcome

Design the `country_profile` table and prepare the production ETL script.

## Questions to answer per project
Just focus on the World Bank dataset. By the end of the session, aim to have answered these four questions:

- What data is available?
- What columns will SMP store?
- How will the ETL transform the raw data?
- What will the PostgreSQL table look like?

In [1]:
# Libraries
import pandas as pd
import requests
from datetime import datetime

# Countries
COUNTRIES = {
    "NGA": "Nigeria",
    "GHA": "Ghana",
    "KEN": "Kenya"
}



In [2]:
# Indicators
INDICATORS = {
    "gdp_current_usd": "NY.GDP.MKTP.CD",
    "gdp_growth": "NY.GDP.MKTP.KD.ZG",
    "population": "SP.POP.TOTL"
}


# World bank API Func
BASE_URL = "https://api.worldbank.org/v2"


def fetch_indicator(country_code, indicator_code):
    """
    Fetch indicator data from the World Bank API.
    """

    url = (
        f"{BASE_URL}/country/{country_code}/indicator/"
        f"{indicator_code}?format=json&per_page=100"
    )

    response = requests.get(url)

    if response.status_code != 200:
        print(f"Error fetching data for {country_code}")
        return []

    data = response.json()

    if len(data) < 2:
        return []

    return data[1]

In [3]:
# Test the API
gdp = fetch_indicator("NGA", "NY.GDP.MKTP.CD")

print(type(gdp))
print(len(gdp))

<class 'list'>
66


In [6]:
# Inspect one record
gdp[0]

{'indicator': {'id': 'NY.GDP.MKTP.CD', 'value': 'GDP (current US$)'},
 'country': {'id': 'NG', 'value': 'Nigeria'},
 'countryiso3code': 'NGA',
 'date': '2025',
 'value': 290794361542.112,
 'unit': '',
 'obs_status': '',
 'decimal': 0}

In [7]:
gdp[-1]

{'indicator': {'id': 'NY.GDP.MKTP.CD', 'value': 'GDP (current US$)'},
 'country': {'id': 'NG', 'value': 'Nigeria'},
 'countryiso3code': 'NGA',
 'date': '1960',
 'value': 4196174501.5302,
 'unit': '',
 'obs_status': '',
 'decimal': 0}

In [8]:
# Create the ETL
records = []

for country_code, country_name in COUNTRIES.items():

    for metric_name, indicator_code in INDICATORS.items():

        print(f"Fetching {metric_name} for {country_name}...")

        data = fetch_indicator(country_code, indicator_code)

        for row in data:

            records.append({

                "country_code": country_code,
                "country": country_name,

                "indicator": metric_name,

                "year": row["date"],

                "value": row["value"]

            })

Fetching gdp_current_usd for Nigeria...
Fetching gdp_growth for Nigeria...
Fetching population for Nigeria...
Fetching gdp_current_usd for Ghana...
Fetching gdp_growth for Ghana...
Fetching population for Ghana...
Fetching gdp_current_usd for Kenya...
Fetching gdp_growth for Kenya...
Fetching population for Kenya...


In [9]:
# Convert to DF
df = pd.DataFrame(records)

df.head()

,country_code,country,indicator,year,value
0,NGA,Nigeria,gdp_current_usd,2025,2.907944e+11
1,NGA,Nigeria,gdp_current_usd,2024,2.522619e+11
2,NGA,Nigeria,gdp_current_usd,2023,4.873878e+11
3,NGA,Nigeria,gdp_current_usd,2022,6.469503e+11
4,NGA,Nigeria,gdp_current_usd,2021,6.091477e+11


In [10]:
# Check the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 594 entries, 0 to 593
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   country_code  594 non-null    object 
 1   country       594 non-null    object 
 2   indicator     594 non-null    object 
 3   year          594 non-null    object 
 4   value         591 non-null    float64
dtypes: float64(1), object(4)
memory usage: 23.3+ KB


In [11]:
# Describe it
df.describe(include="all")

,country_code,country,indicator,year,value
count,594,594,594,594,5.910000e+02
unique,3,3,3,66,NaN
top,NGA,Nigeria,gdp_current_usd,2025,NaN
freq,198,198,198,9,NaN
mean,NaN,NaN,NaN,NaN,2.504564e+10
std,NaN,NaN,NaN,NaN,8.581015e+10
min,NaN,NaN,NaN,NaN,-1.574363e+01
25%,NaN,NaN,NaN,NaN,6.201631e+00
50%,NaN,NaN,NaN,NaN,3.125894e+07
75%,NaN,NaN,NaN,NaN,5.181678e+09


In [12]:
# Check for Null Vals
df.isnull().sum()

country_code    0
country         0
indicator       0
year            0
value           3
dtype: int64

In [13]:
# Remove missing vals
df = df.dropna(subset=["value"])

df.head()

,country_code,country,indicator,year,value
0,NGA,Nigeria,gdp_current_usd,2025,2.907944e+11
1,NGA,Nigeria,gdp_current_usd,2024,2.522619e+11
2,NGA,Nigeria,gdp_current_usd,2023,4.873878e+11
3,NGA,Nigeria,gdp_current_usd,2022,6.469503e+11
4,NGA,Nigeria,gdp_current_usd,2021,6.091477e+11


In [14]:
# Check again
# Check for Null Vals
df.isnull().sum()

country_code    0
country         0
indicator       0
year            0
value           0
dtype: int64

In [15]:
# Convert data types
df["year"] = df["year"].astype(int)
df["value"] = df["value"].astype(float)

In [16]:
# Check data types
df.dtypes

country_code     object
country          object
indicator        object
year              int64
value           float64
dtype: object

In [17]:
# Sort data
df = df.sort_values(
    by=["country", "indicator", "year"],
    ascending=[True, True, False]
)

In [18]:
# Preview
df.head(20)

,country_code,country,indicator,year,value
198,GHA,Ghana,gdp_current_usd,2025,1.142099e+11
199,GHA,Ghana,gdp_current_usd,2024,8.328859e+10
200,GHA,Ghana,gdp_current_usd,2023,8.054715e+10
201,GHA,Ghana,gdp_current_usd,2022,7.391900e+10
202,GHA,Ghana,gdp_current_usd,2021,7.951420e+10
203,GHA,Ghana,gdp_current_usd,2020,7.000824e+10
204,GHA,Ghana,gdp_current_usd,2019,6.835263e+10
205,GHA,Ghana,gdp_current_usd,2018,6.725935e+10
206,GHA,Ghana,gdp_current_usd,2017,6.038541e+10
207,GHA,Ghana,gdp_current_usd,2016,5.614418e+10


In [19]:
# Save as Staging data
df.to_csv(
    "country_profile_raw.csv",
    index=False
)

print("Saved successfully!")

Saved successfully!


In [20]:
# Transform for Production table
country_profile = (
    df.pivot_table(
        index=["country_code", "country", "year"],
        columns="indicator",
        values="value",
        aggfunc="first"
    )
    .reset_index()
)

# Remove index
country_profile.columns.name = None

# Check prod data
country_profile.head()

,country_code,country,year,gdp_current_usd,gdp_growth,population
0,GHA,Ghana,1960,1.217230e+09,NaN,6961215.0
1,GHA,Ghana,1961,1.302674e+09,3.429674,7162667.0
2,GHA,Ghana,1962,1.382516e+09,4.109159,7337375.0
3,GHA,Ghana,1963,1.540798e+09,4.405974,7514714.0
4,GHA,Ghana,1964,1.731296e+09,2.209328,7695739.0


In [21]:
# Validate
country_profile.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 198 entries, 0 to 197
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   country_code     198 non-null    object 
 1   country          198 non-null    object 
 2   year             198 non-null    int64  
 3   gdp_current_usd  198 non-null    float64
 4   gdp_growth       195 non-null    float64
 5   population       198 non-null    float64
dtypes: float64(3), int64(1), object(2)
memory usage: 9.4+ KB


In [22]:
# Check for null vals
country_profile.isnull().sum()

country_code       0
country            0
year               0
gdp_current_usd    0
gdp_growth         3
population         0
dtype: int64

In [24]:
# Remove null vals
country_profile = country_profile.dropna(subset=["gdp_growth"])

In [25]:
# Check again
country_profile.isnull().sum()

country_code       0
country            0
year               0
gdp_current_usd    0
gdp_growth         0
population         0
dtype: int64

In [26]:
# Check prod data
country_profile.head()

,country_code,country,year,gdp_current_usd,gdp_growth,population
1,GHA,Ghana,1961,1.302674e+09,3.429674,7162667.0
2,GHA,Ghana,1962,1.382516e+09,4.109159,7337375.0
3,GHA,Ghana,1963,1.540798e+09,4.405974,7514714.0
4,GHA,Ghana,1964,1.731296e+09,2.209328,7695739.0
5,GHA,Ghana,1965,2.053463e+09,1.368999,7882606.0


In [27]:
# Save
df.to_csv("raw_country_profile.csv", index=False)

In [28]:
# Save curated
country_profile.to_csv(
    "country_profile.csv",
    index=False
)